# AlegroCode — Jupyter / rented GPU launcher

Non-blocking FastAPI server launched right inside the notebook kernel via `nest_asyncio`.
Cells return control immediately so you can keep working in the notebook while the API is live.

**Order to run:** 1 → 2 → 3 → 4 → 5. Cell 6 is the graceful shutdown.

## Cell 1 — Setup
Install Python deps and enable nested event loops.

In [ ]:
%pip install -q -r backend/requirements.txt
%pip install -q git+https://github.com/facebookresearch/sam2.git || true

import os, sys
repo_root = os.path.abspath(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import nest_asyncio
nest_asyncio.apply()
print('nest_asyncio applied — Jupyter event loop is now reentrant.')

## Cell 2 — Load models ONCE
Models live in kernel globals — restarting the server in Cell 3 does NOT re-download weights.

In [ ]:
os.environ.setdefault('ALEGRO_INPAINT_PROVIDER', 'lama')
# os.environ['ALEGRO_NGROK_AUTHTOKEN'] = 'YOUR-TOKEN-HERE'

from backend.api import create_app
from backend.ml_pipeline import FacadeAnalyzer

analyzer = FacadeAnalyzer()
analyzer.load_models()
app = create_app(analyzer=analyzer)
print('Analyzer ready on device:', analyzer.device)

## Cell 3 — Start the server (non-blocking)
`asyncio.create_task(server.serve())` schedules uvicorn on the existing Jupyter loop and returns immediately.

In [ ]:
import asyncio, uvicorn

config = uvicorn.Config(app, host='0.0.0.0', port=8000, loop='asyncio', log_level='info')
server = uvicorn.Server(config)
server_task = asyncio.ensure_future(server.serve())
print('Server task scheduled. The cell is released — API is booting in the background.')

## Cell 4 — Health check
Wait until the API is actually listening.

In [ ]:
import httpx, time

for _ in range(30):
    try:
        r = httpx.get('http://127.0.0.1:8000/api/health', timeout=2)
        if r.status_code == 200:
            print('healthy:', r.json())
            break
    except httpx.RequestError:
        pass
    time.sleep(1)
else:
    raise RuntimeError('server failed to come up in 30s')

## Cell 5 — Expose via ngrok tunnel
Fill `ngrok_token` with your authtoken. Public URL is read from `.public_url` directly, not a regex.

In [ ]:
from pyngrok import ngrok

ngrok_token = os.environ.get('ALEGRO_NGROK_AUTHTOKEN', '')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

tunnel = ngrok.connect(8000, 'http')
print('='*60)
print('PUBLIC URL:', tunnel.public_url)
print('DOCS      :', tunnel.public_url + '/docs')
print('HEALTH    :', tunnel.public_url + '/api/health')
print('='*60)
print('Paste the PUBLIC URL into the Flutter app Settings screen.')

## Cell 6 — Graceful shutdown
Run this to stop the server without killing the kernel; models stay loaded so Cell 3 can re-start instantly.

In [ ]:
server.should_exit = True
try:
    await asyncio.wait_for(server_task, timeout=10)
except asyncio.TimeoutError:
    server_task.cancel()
try:
    ngrok.disconnect(tunnel.public_url)
except Exception as e:
    print('ngrok.disconnect:', e)
ngrok.kill()
print('Server stopped. Models remain loaded in memory.')

## Cell 7 (optional) — Run the price scraper once
APScheduler already runs nightly when `ALEGRO_SCRAPER_ENABLED=true`. This cell forces a one-off run.

In [ ]:
from backend.scraper.worker import run_once
await run_once('all')